# Data Cleaning and Validation — AI4I Predictive Maintenance Analytics

Validate the original CSV before creating the dataset used in later notebooks.

**Input:** `data/ai4i2020.csv`  
**Outputs created when this notebook runs:**
- `data/cleaned/ai4i2020_cleaned.csv`
- `data/cleaned/data_quality_report.json`
- `data/cleaned/failure_label_review.csv`

This adapts the [original capstone cleaning notebook](https://github.com/roshantuladhar00/Capstone_Project_Damo-699/blob/dev/notebook/03_data_cleaning.ipynb) to this project's existing CSV location, removes the dependency on custom modules, and adds a review of the target/flag disagreements found during EDA.

This notebook validates and preserves the supplied labels. It does not infer replacement labels from the failure flags. The original source license is included at the end.


## 1. Load the original dataset

Find the existing `data/ai4i2020.csv` from the project root or its `notebook` folder. Keep the original file untouched and record its checksum so this can be verified at the end.


In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in (current, *current.parents)
     if (folder / "data" / "ai4i2020.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot find data/ai4i2020.csv. Open this notebook inside the project folder."
    )

RAW_PATH = PROJECT_ROOT / "data" / "ai4i2020.csv"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

raw_sha256_before = sha256_file(RAW_PATH)
df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy(deep=True)

print(f"Input: {RAW_PATH}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(df.head())


Input: C:\Users\rosha\Documents\Codex\2026-10-16\how\work\cleaning_validation\data\ai4i2020.csv
Shape: 10,000 rows x 14 columns


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 2. Validate columns and types

Check the expected sensor inputs, identifiers, product type, overall failure label, and five failure flags. Unexpected missing columns or nonnumeric sensor/label values stop execution so they are investigated rather than silently converted.


In [2]:
SENSOR_COLUMNS = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
FAILURE_FLAGS = ["TWF", "HDF", "PWF", "OSF", "RNF"]
TARGET = "Machine failure"
EXPECTED_COLUMNS = ["UDI", "Product ID", "Type", *SENSOR_COLUMNS, TARGET, *FAILURE_FLAGS]

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
if missing_columns:
    raise ValueError(f"Required columns missing from the CSV: {missing_columns}")
if df.empty:
    raise ValueError("The CSV contains no observations.")

NUMERIC_COLUMNS = ["UDI", *SENSOR_COLUMNS, TARGET, *FAILURE_FLAGS]
nonnumeric_columns = [
    column for column in NUMERIC_COLUMNS
    if not pd.api.types.is_numeric_dtype(df[column])
]
if nonnumeric_columns:
    raise TypeError(f"Expected numeric data in: {nonnumeric_columns}")

extra_columns = sorted(set(df.columns) - set(EXPECTED_COLUMNS))
print(f"Additional columns retained: {extra_columns}")
display(df.dtypes.astype(str).rename("dtype").to_frame())


Additional columns retained: []


,dtype
UDI,int64
Product ID,object
Type,object
Air temperature [K],float64
Process temperature [K],float64
Rotational speed [rpm],int64
Torque [Nm],float64
Tool wear [min],int64
Machine failure,int64
TWF,int64


## 3. Check missing values and duplicate records

Count missing values in every column, exact duplicate rows, and repeated identifiers. Nothing is dropped or imputed automatically. The export step will stop if a required validation fails.


In [3]:
missing_by_column = df.isna().sum()
duplicate_rows = int(df.duplicated().sum())
duplicate_udi = int(df["UDI"].duplicated().sum())
duplicate_product_id = int(df["Product ID"].duplicated().sum())

display(missing_by_column.rename("missing_count").to_frame())
display(pd.Series({
    "exact_duplicate_rows": duplicate_rows,
    "repeated_UDI_values": duplicate_udi,
    "repeated_Product_ID_values": duplicate_product_id,
}, name="count").to_frame())

numeric_values = df[NUMERIC_COLUMNS].to_numpy(dtype=float)
nonfinite_count = int((~np.isfinite(numeric_values)).sum())
print(f"Nonfinite numeric entries (including missing numeric entries): {nonfinite_count}")


,missing_count
UDI,0
Product ID,0
Type,0
Air temperature [K],0
Process temperature [K],0
Rotational speed [rpm],0
Torque [Nm],0
Tool wear [min],0
Machine failure,0
TWF,0


,count
exact_duplicate_rows,0
repeated_UDI_values,0
repeated_Product_ID_values,0


Nonfinite numeric entries (including missing numeric entries): 0


## 4. Validate the target, failure flags, and product type

The supplied overall target and failure flags must contain only 0 and 1. Type must be L, M, or H. Blank identifiers and surrounding whitespace are reported as invalid; they are not silently rewritten.


In [4]:
invalid_binary_counts = {
    column: int((~df[column].isin([0, 1])).sum())
    for column in [TARGET, *FAILURE_FLAGS]
}
invalid_type_count = int((~df["Type"].isin(["L", "M", "H"])).sum())

product_id_text = df["Product ID"].astype("string")
invalid_product_id_count = int(
    (product_id_text.isna()
     | product_id_text.fillna("").str.strip().eq("")
     | product_id_text.ne(product_id_text.str.strip()).fillna(False)).sum()
)

invalid_udi_count = int((
    df["UDI"].isna()
    | (df["UDI"] <= 0)
    | (df["UDI"] % 1 != 0)
).sum())

display(pd.Series(invalid_binary_counts, name="invalid_binary_count").to_frame())
print(f"Invalid Type values: {invalid_type_count}")
print(f"Blank or whitespace-padded Product IDs: {invalid_product_id_count}")
print(f"Nonpositive, missing, or noninteger UDI values: {invalid_udi_count}")

target_counts = df[TARGET].value_counts(dropna=False).sort_index()
display(pd.DataFrame({
    "count": target_counts,
    "percentage": target_counts / len(df) * 100,
}).round(2))


,invalid_binary_count
Machine failure,0
TWF,0
HDF,0
PWF,0
OSF,0
RNF,0


Invalid Type values: 0
Blank or whitespace-padded Product IDs: 0
Nonpositive, missing, or noninteger UDI values: 0


,count,percentage
Machine failure,,
0,9661,96.61
1,339,3.39


## 5. Inspect sensor values and unusual observations

Check basic constraints: temperatures in kelvin and rotational speed should be positive; torque and tool wear should be nonnegative. These are basic plausibility checks, not a full engineering validation of operating limits.

The 1.5-IQR rule below identifies unusual readings for review. It does not prove a reading is erroneous, so these rows are retained. No clipping, scaling, resampling, or feature engineering is applied during this cleaning step.


In [5]:
sensor_invalid_counts = {
    "Air temperature <= 0": int((df["Air temperature [K]"] <= 0).sum()),
    "Process temperature <= 0": int((df["Process temperature [K]"] <= 0).sum()),
    "Rotational speed <= 0": int((df["Rotational speed [rpm]"] <= 0).sum()),
    "Torque < 0": int((df["Torque [Nm]"] < 0).sum()),
    "Tool wear < 0": int((df["Tool wear [min]"] < 0).sum()),
}
display(pd.Series(sensor_invalid_counts, name="invalid_count").to_frame())
display(df[SENSOR_COLUMNS].agg(["min", "median", "max"]).T)

outlier_records = []
for column in SENSOR_COLUMNS:
    q1, q3 = df[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = (df[column] < lower) | (df[column] > upper)
    outlier_records.append({
        "feature": column,
        "lower_IQR_fence": float(lower),
        "upper_IQR_fence": float(upper),
        "flagged_rows": int(flagged.sum()),
        "flagged_rows_with_failure": int(df.loc[flagged, TARGET].sum()),
    })
outlier_summary = pd.DataFrame(outlier_records).set_index("feature")
display(outlier_summary)
print("IQR-flagged observations are retained.")


,invalid_count
Air temperature <= 0,0
Process temperature <= 0,0
Rotational speed <= 0,0
Torque < 0,0
Tool wear < 0,0


,min,median,max
Air temperature [K],295.3,300.1,304.5
Process temperature [K],305.7,310.1,313.8
Rotational speed [rpm],1168.0,1503.0,2886.0
Torque [Nm],3.8,40.1,76.6
Tool wear [min],0.0,108.0,253.0


,lower_IQR_fence,upper_IQR_fence,flagged_rows,flagged_rows_with_failure
feature,,,,
Air temperature [K],293.50,306.30,0,0
Process temperature [K],305.35,314.55,0,0
Rotational speed [rpm],1139.50,1895.50,418,35
Torque [Nm],12.80,67.20,69,62
Tool wear [min],-110.50,325.50,0,0


IQR-flagged observations are retained.


## 6. Compare the overall target with individual failure flags

Compare `Machine failure` with whether at least one failure flag is 1. Also count overlapping failure flags.

A disagreement identifies a record for review; it does not establish which field should be corrected. Preserve all supplied labels and rows and export the disagreeing records separately. This records an explicit policy rather than guessing replacement labels.


In [6]:
flags_per_row = df[FAILURE_FLAGS].sum(axis=1)
any_failure_flag = df[FAILURE_FLAGS].eq(1).any(axis=1)
label_disagreement = any_failure_flag.ne(df[TARGET].eq(1))

failure_without_flag = int((df[TARGET].eq(1) & ~any_failure_flag).sum())
flag_without_failure = int((df[TARGET].eq(0) & any_failure_flag).sum())
overlapping_flags = int(flags_per_row.gt(1).sum())

display(pd.crosstab(
    df[TARGET],
    any_failure_flag,
    rownames=["Machine failure"],
    colnames=["Any failure flag"],
))
print(f"Overall failure = 1, but no failure flag = 1: {failure_without_flag}")
print(f"Overall failure = 0, but at least one flag = 1: {flag_without_failure}")
print(f"Total disagreements: {int(label_disagreement.sum())}")
print(f"Rows with multiple active failure flags: {overlapping_flags}")

review_columns = ["UDI", "Product ID", "Type", TARGET, *FAILURE_FLAGS]
label_review = df.loc[label_disagreement, review_columns].copy()
label_review["any_failure_flag"] = any_failure_flag.loc[label_disagreement].astype(int)
label_review["review_reason"] = np.where(
    label_review[TARGET].eq(1),
    "target_1_without_failure_flag",
    "target_0_with_failure_flag",
)
display(label_review.head(10))


Any failure flag,False,True
Machine failure,,
0,9643,18
1,9,330


Overall failure = 1, but no failure flag = 1: 9
Overall failure = 0, but at least one flag = 1: 18
Total disagreements: 27
Rows with multiple active failure flags: 24


,UDI,Product ID,Type,Machine failure,TWF,HDF,PWF,OSF,RNF,any_failure_flag,review_reason
1221,1222,M16081,M,0,0,0,0,0,1,1,target_0_with_failure_flag
1302,1303,L48482,L,0,0,0,0,0,1,1,target_0_with_failure_flag
1437,1438,H30851,H,1,0,0,0,0,0,0,target_1_without_failure_flag
1748,1749,H31162,H,0,0,0,0,0,1,1,target_0_with_failure_flag
2072,2073,L49252,L,0,0,0,0,0,1,1,target_0_with_failure_flag
2559,2560,L49739,L,0,0,0,0,0,1,1,target_0_with_failure_flag
2749,2750,M17609,M,1,0,0,0,0,0,0,target_1_without_failure_flag
3065,3066,M17925,M,0,0,0,0,0,1,1,target_0_with_failure_flag
3452,3453,H32866,H,0,0,0,0,0,1,1,target_0_with_failure_flag
4044,4045,M18904,M,1,0,0,0,0,0,0,target_1_without_failure_flag


## 7. Apply the cleaning policy

For this dataset, cleaning consists of validation and a documented export. The notebook checks whether the data actually satisfy the conditions below before saving:

- Required columns exist and numeric fields have numeric types.
- Missing values, duplicate records/identifiers, nonfinite numbers, invalid categories, invalid binary labels, and basic sensor constraint violations must be investigated before export.
- Label/flag disagreements and IQR flags are retained and reported.
- The supplied target remains unchanged.
- Identifier and failure-flag columns are retained in the audit dataset. Exclude them from future model inputs.


In [7]:
validation_counts = {
    "missing_values": int(missing_by_column.sum()),
    "exact_duplicate_rows": duplicate_rows,
    "repeated_UDI_values": duplicate_udi,
    "repeated_Product_ID_values": duplicate_product_id,
    "nonfinite_numeric_entries": nonfinite_count,
    "invalid_type_values": invalid_type_count,
    "invalid_Product_ID_values": invalid_product_id_count,
    "invalid_UDI_values": invalid_udi_count,
    **{f"invalid_binary_{column}": count for column, count in invalid_binary_counts.items()},
    **sensor_invalid_counts,
}
blocking_issues = {
    check: count for check, count in validation_counts.items()
    if count > 0
}
if blocking_issues:
    raise ValueError(
        "Export stopped. Inspect these issues and decide a cleaning rule first: "
        + json.dumps(blocking_issues)
    )

df_cleaned = df.copy(deep=True)
pd.testing.assert_frame_equal(df_cleaned, df_raw)
print("All required validation checks passed.")
print(f"Rows retained: {len(df_cleaned):,} / {len(df_raw):,}")
print(f"Columns retained: {df_cleaned.shape[1]}")
print("No observations, sensor values, identifiers, or target labels were changed.")


All required validation checks passed.
Rows retained: 10,000 / 10,000
Columns retained: 14
No observations, sensor values, identifiers, or target labels were changed.


## 8. Save the validated dataset and review report

Create `data/cleaned` automatically. The output CSV preserves the original column order and values. The quality report records the input checksum and cleaning policy, and the review CSV contains only the target/flag disagreements.


In [8]:
if sha256_file(RAW_PATH) != raw_sha256_before:
    raise RuntimeError("The source CSV changed during this run. Restart and rerun the notebook.")

CLEANED_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_PATH = CLEANED_DIR / "ai4i2020_cleaned.csv"
QUALITY_REPORT_PATH = CLEANED_DIR / "data_quality_report.json"
LABEL_REVIEW_PATH = CLEANED_DIR / "failure_label_review.csv"

quality_report = {
    "input_relative_path": RAW_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "input_sha256": raw_sha256_before,
    "cleaned_relative_path": CLEANED_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "rows_before": int(len(df_raw)),
    "rows_after": int(len(df_cleaned)),
    "columns_after": int(df_cleaned.shape[1]),
    "validation_issue_counts": validation_counts,
    "machine_failure_counts": {
        str(int(label)): int(count)
        for label, count in df_cleaned[TARGET].value_counts().sort_index().items()
    },
    "failure_rate": float(df_cleaned[TARGET].mean()),
    "label_review": {
        "total_disagreements": int(label_disagreement.sum()),
        "failure_without_flag": failure_without_flag,
        "flag_without_failure": flag_without_failure,
        "rows_with_overlapping_flags": overlapping_flags,
        "policy": "Preserve the supplied Machine failure label and all failure flags.",
    },
    "IQR_flags": outlier_records,
    "cleaning_policy": {
        "rows_removed": 0,
        "imputation_applied": False,
        "sensor_values_clipped": False,
        "labels_rewritten": False,
        "resampling_applied": False,
        "scaling_applied": False,
        "engineered_features_added": False,
    },
    "exclude_from_future_model_inputs": ["UDI", "Product ID", *FAILURE_FLAGS],
}

df_cleaned.to_csv(CLEANED_PATH, index=False)
label_review.to_csv(LABEL_REVIEW_PATH, index=False)
QUALITY_REPORT_PATH.write_text(
    json.dumps(quality_report, indent=2, allow_nan=False),
    encoding="utf-8",
)

# Check the actual saved CSV, including labels and row/column order.
saved_df = pd.read_csv(CLEANED_PATH)
pd.testing.assert_frame_equal(saved_df, df_cleaned)
if sha256_file(RAW_PATH) != raw_sha256_before:
    raise RuntimeError("The raw CSV checksum changed unexpectedly.")

for path in [CLEANED_PATH, QUALITY_REPORT_PATH, LABEL_REVIEW_PATH]:
    print(f"Saved: {path.relative_to(PROJECT_ROOT)}")
print("Verified: raw file unchanged and saved CSV matches the validated dataframe.")


Saved: data\cleaned\ai4i2020_cleaned.csv
Saved: data\cleaned\data_quality_report.json
Saved: data\cleaned\failure_label_review.csv
Verified: raw file unchanged and saved CSV matches the validated dataframe.


## 9. Results from this run

The following summary is calculated from the loaded dataset. Validation is not a claim that every label is correct: target/flag disagreements remain documented for later analysis.


In [9]:
summary = pd.Series({
    "input_rows": len(df_raw),
    "cleaned_rows": len(saved_df),
    "removed_rows": len(df_raw) - len(saved_df),
    "missing_values": int(saved_df.isna().sum().sum()),
    "exact_duplicate_rows": int(saved_df.duplicated().sum()),
    "machine_failures": int(saved_df[TARGET].sum()),
    "failure_rate_percent": round(float(saved_df[TARGET].mean()) * 100, 2),
    "target_flag_disagreements_retained": int(label_disagreement.sum()),
}, name="value")
display(summary.to_frame())
print("Cleaned dataset is ready for statistical analysis and model preparation.")
print("Keep identifiers and failure-type flags out of the prediction feature matrix.")


,value
input_rows,10000.00
cleaned_rows,10000.00
removed_rows,0.00
missing_values,0.00
exact_duplicate_rows,0.00
machine_failures,339.00
failure_rate_percent,3.39
target_flag_disagreements_retained,27.00


Cleaned dataset is ready for statistical analysis and model preparation.
Keep identifiers and failure-type flags out of the prediction feature matrix.


## Original source license

Original MIT notice for the capstone material adapted in this notebook:

```text
MIT License

Copyright (c) 2026 Faizanfarid Malek

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

```
